# H22a: Anisotropy Sweet Spot -- Muon Benefit Peaks at Moderate Effective Rank

## Theoretical Motivation

The Newton-Schulz step in Muon replaces $G = U\Sigma V^T$ with $UV^T$, equalizing all
singular values to 1. But this equalization is not equally useful for all gradient shapes:

- **Too isotropic** ($\sigma_1 \approx \sigma_{\min}$, high effective rank): The gradient
  is already nearly a scaled partial isometry. Newton-Schulz barely changes it, so
  Muon $\approx$ SGD. No room for improvement.

- **Too anisotropic** ($\sigma_1 \gg \sigma_{\min}$, low effective rank): The gradient
  is nearly rank-1. Newton-Schulz maps it to a partial isometry, but this "lifts"
  the noise singular values to the same magnitude as the signal direction. The update
  wastes most of its budget on noise directions that carry no loss information.

- **Moderate effective rank** (30-70% of full rank): There is meaningful structure in
  multiple singular directions, but they are imbalanced. Newton-Schulz equalizes them,
  letting the optimizer use ALL informative directions equally. This is where Muon's
  spectral democracy yields maximum benefit.

## Hypothesis

> **H22a:** Muon's advantage over SGD shows an **inverted-U shape** as a function of
> gradient effective rank fraction. The peak occurs at $\text{eff\_rank}/\text{dim} \in [0.2, 0.8]$,
> and the advantage is lower at both extremes.

## Why This Matters

This is a **non-trivial prediction** of the spectral equalization theory. Most theories
of optimization predict monotonic relationships (more ill-conditioning = more benefit from
preconditioning). The inverted-U prediction arises because Muon's Newton-Schulz step is
not a preconditioner -- it is a **projection** that discards all scale information. This
projection helps when there is structure to equalize, but hurts when it amplifies noise
directions.

## Experimental Design

We control gradient effective rank indirectly by constructing target matrices with
power-law singular value spectra: $\sigma_i = i^{-\alpha}$ for $i = 1, \ldots, 32$.

| $\alpha$ | Spectrum Shape     | Expected Eff. Rank | Expected Muon Benefit |
|----------|--------------------|--------------------|-----------------------|
| 0.1      | Nearly flat        | ~95% (isotropic)   | Low (already balanced)|
| 0.3      | Gentle decay       | ~80%               | Moderate              |
| 0.5      | Moderate decay     | ~60%               | **High (sweet spot)** |
| 1.0      | Zipf's law         | ~40%               | **High (sweet spot)** |
| 2.0      | Steep decay        | ~25%               | Moderate              |
| 5.0      | Very steep         | ~10%               | Low (near rank-1)    |
| 10.0     | Near rank-1        | ~5%                | Low (noise dominates)|

**Network:** 4-layer deep linear, 32x32, trained 500 steps. Both optimizers get
independent LR sweeps (12-point log grid from $10^{-4}$ to $10^{-1}$).

## Environment Setup

Pure NumPy implementation. The deep linear network has no activations, so the gradient
computation is a simple chain of matrix products -- fully transparent and auditable.

In [ ]:
import numpy as np
import os

In [ ]:
SCRIPT_DIR = os.path.dirname(os.path.abspath('.'))


## Experimental Configuration

Key hyperparameters:

- **DIM = 32**: Weight matrices are 32x32, giving 32 singular values to control.
- **NUM_LAYERS = 4**: Deep linear network -- the product of layers creates a
  nontrivial end-to-end map whose gradient spectrum reflects both the target
  anisotropy and the depth-induced spectral distortion.
- **NUM_STEPS = 500**: Training duration -- long enough for convergence patterns
  to emerge but short enough for the LR sweep to be tractable.
- **MOMENTUM = 0.9**: Identical for SGD and Muon.
- **NS_ITERS = 5**: Newton-Schulz iterations.
- **NUM_SEEDS = 5**: Independent random seeds for statistical averaging.
- **BATCH_SIZE = 64**: Fixed batch (no stochastic noise).

In [ ]:
DIM = 32
NUM_LAYERS = 4
NUM_STEPS = 500
MOMENTUM = 0.9
NS_ITERS = 5
NUM_SEEDS = 5
BATCH_SIZE = 64

print("\n--- Experimental Configuration ---")
print(f"  DIM            = {DIM}    (weight matrix dimension)")
print(f"  NUM_LAYERS     = {NUM_LAYERS}     (deep linear network depth)")
print(f"  NUM_STEPS      = {NUM_STEPS}   (training iterations per run)")
print(f"  BATCH_SIZE     = {BATCH_SIZE}    (fixed batch, no stochastic noise)")
print(f"  MOMENTUM       = {MOMENTUM}  (heavy-ball, same for SGD and Muon)")
print(f"  NUM_SEEDS      = {NUM_SEEDS}     (independent random initializations)")
print(f"  NS_ITERS       = {NS_ITERS}     (Newton-Schulz iterations for Muon)")

### Anisotropy Control via Power-Law Exponent

The key experimental variable is $\alpha$, the power-law exponent controlling target
matrix singular values: $\sigma_i = i^{-\alpha}$. Small $\alpha$ gives a flat spectrum
(high effective rank, isotropic); large $\alpha$ gives a steep spectrum (low effective
rank, anisotropic). This spans the full range from "already democratic" to "extremely
concentrated" gradient geometries.

In [ ]:
# Control anisotropy via power-law exponent on target matrix SVs
ALPHA_VALUES = [0.1, 0.3, 0.5, 1.0, 2.0, 5.0, 10.0]

In [ ]:
LR_SGD = np.logspace(-4, -1, 12)
LR_MUON = np.logspace(-4, -1, 12)

## Core Functions: Newton-Schulz, Effective Rank, and Target Construction

**Newton-Schulz:** Computes the polar factor $UV^T$ of a gradient matrix (Muon's core operation).

**Effective rank:** Shannon entropy-based measure of spectral spread:
$\text{eff\_rank}(M) = \exp\left(-\sum_i p_i \log p_i\right)$ where $p_i = \sigma_i^2 / \sum_j \sigma_j^2$.
This measures "how many singular values are significantly nonzero" on a continuous scale
from 1 (rank-1) to $\min(m,n)$ (uniform spectrum).

**Anisotropic target:** Constructs $T = U \cdot \text{diag}(\sigma) \cdot V^T$ with
$\sigma_i = i^{-\alpha}$, normalized so $\|T\|_F = \sqrt{d}$. Random orthogonal $U, V$
ensure the anisotropy is in the spectral structure, not aligned with coordinate axes.

In [ ]:
def newton_schulz(M, n_iters=NS_ITERS):
    norm = np.linalg.norm(M, 'fro')
    if norm < 1e-15:
        return M
    X = M / norm
    for _ in range(n_iters):
        A = X.T @ X
        X = 1.5 * X - 0.5 * X @ A
    return X

In [ ]:
def effective_rank(M):
    """Compute effective rank = exp(entropy of normalized SV^2)."""
    sv = np.linalg.svd(M, compute_uv=False)
    sv2 = sv**2
    sv2 = sv2 / (np.sum(sv2) + 1e-30)
    sv2 = sv2[sv2 > 1e-30]
    entropy = -np.sum(sv2 * np.log(sv2))
    return np.exp(entropy)

In [ ]:
def make_anisotropic_target(alpha, seed):
    """Create target matrix with power-law SVs: sigma_i = i^(-alpha)."""
    rng = np.random.RandomState(seed)
    U, _ = np.linalg.qr(rng.randn(DIM, DIM))
    V, _ = np.linalg.qr(rng.randn(DIM, DIM))
    sigmas = np.array([(i + 1)**(-alpha) for i in range(DIM)])
    sigmas = sigmas / np.linalg.norm(sigmas) * np.sqrt(DIM)  # normalize
    return U @ np.diag(sigmas) @ V.T

**Weight initialization:** $W_l = I + 0.1 \cdot \mathcal{N}(0,1)$ -- near-identity init so
the network starts close to the identity map. The gradient spectrum is then dominated by
the target matrix's spectral structure, not the random initialization.

In [ ]:
def init_weights(seed):
    rng = np.random.RandomState(seed)
    return [np.eye(DIM) + rng.randn(DIM, DIM) * 0.1 for _ in range(NUM_LAYERS)]

## Network: Deep Linear Forward Pass, Loss, and Gradients

The deep linear network computes $\hat{y} = W_4 W_3 W_2 W_1 x$ (no activations).
The MSE loss is $\mathcal{L} = \frac{1}{2N}\sum_n \|\hat{y}_n - y_n\|^2$.
Gradients are computed by explicit backpropagation through the linear chain.

**Why deep linear?** In a deep linear network, the end-to-end map is $\prod_l W_l$,
which must approximate the target matrix $T$. The gradient spectrum at each layer
inherits structure from $T$'s singular values, giving us precise control over the
anisotropy the optimizer encounters.

In [ ]:
def forward(weights, X):
    out = X.copy()
    for W in weights:
        out = W @ out
    return out

In [ ]:
def compute_loss(weights, X, Y):
    pred = forward(weights, X)
    return 0.5 * np.mean(np.sum((pred - Y)**2, axis=0))

In [ ]:
def compute_gradients(weights, X, Y):
    L = len(weights)
    N = X.shape[1]
    acts = [X.copy()]
    for W in weights:
        acts.append(W @ acts[-1])
    delta = (acts[-1] - Y) / N
    grads = [None] * L
    for l in range(L - 1, -1, -1):
        grads[l] = delta @ acts[l].T
        if l > 0:
            delta = weights[l].T @ delta
    return grads

## Training Engine

Standard SGD+momentum / Muon+momentum training loop. Divergence guard at $10^{10}$.
The only difference between optimizers is the Newton-Schulz step applied to the gradient
before momentum accumulation.

In [ ]:
def train(w0, X, Y, lr, opt):
    weights = [W.copy() for W in w0]
    mom = [np.zeros_like(W) for W in weights]
    for step in range(NUM_STEPS):
        if compute_loss(weights, X, Y) > 1e10:
            return float('inf')
        grads = compute_gradients(weights, X, Y)
        for i in range(NUM_LAYERS):
            if opt == 'muon':
                mom[i] = MOMENTUM * mom[i] + newton_schulz(grads[i])
            else:
                mom[i] = MOMENTUM * mom[i] + grads[i]
            weights[i] -= lr * mom[i]
    return compute_loss(weights, X, Y)

### Diagnostic: Target Matrix Spectra Across Alpha Values

Before the main experiment, let us visualize how the power-law exponent $\alpha$ controls
the singular value spectrum and effective rank of the target matrix. This confirms that
our experimental variable spans the intended range from isotropic to near-rank-1.

In [ ]:
print("=" * 80)
print("DIAGNOSTIC: Target Matrix Spectra for Each Alpha")
print("=" * 80)
print(f"\n  {'alpha':>6}  {'sigma_1':>10}  {'sigma_min':>10}  {'ratio':>10}  {'eff_rank':>10}  {'eff_rank%':>10}")
print("  " + "-" * 56)
for alpha in ALPHA_VALUES:
    T = make_anisotropic_target(alpha, seed=42)
    sv = np.linalg.svd(T, compute_uv=False)
    er = effective_rank(T)
    ratio = sv[0] / max(sv[-1], 1e-15)
    print(f"  {alpha:>6.1f}  {sv[0]:>10.4f}  {sv[-1]:>10.6f}  {ratio:>10.1f}  {er:>10.1f}  {er/DIM*100:>9.0f}%")
print(f"\n  Full rank = {DIM} (all SVs equal => eff_rank = {DIM})")
print(f"  Rank 1 (one dominant SV) => eff_rank = 1")
print("\n  The hypothesis predicts Muon advantage peaks at eff_rank ~30-70% of {DIM}")
print("=" * 80)

---
## Main Experiment: Sweep Alpha, Measure Advantage vs. Effective Rank

For each $\alpha$ value:
1. Construct target matrix $T$ with power-law SVs
2. Generate data: $X \sim \mathcal{N}(0, 0.3)$, $Y = TX$
3. Measure gradient effective rank at initialization (averaged over layers and seeds)
4. LR sweep for both SGD and Muon (12 candidates each)
5. Full training at best LR (5 seeds)
6. Compute Muon advantage = $\text{loss}_{\text{SGD}} / \text{loss}_{\text{Muon}}$

The resulting advantage-vs-effective-rank curve should show an inverted-U if the
sweet spot hypothesis is correct.

In [ ]:
def main():
    seeds = [42 + i * 137 for i in range(NUM_SEEDS)]

    print("=" * 100)
    print("H22a: ANISOTROPY SWEET SPOT -- EFFECTIVE RANK PREDICTS MUON BENEFIT")
    print("=" * 100)
    print(f"Alpha (power-law exponent): {ALPHA_VALUES}")
    print(f"Network: {NUM_LAYERS}-layer deep linear {DIM}x{DIM}, {NUM_STEPS} steps")
    print()

    results = {}

    for alpha in ALPHA_VALUES:
        print(f"\n  alpha={alpha}")

        # Measure effective rank of gradient at init
        eff_ranks = []
        for s in seeds[:3]:
            target = make_anisotropic_target(alpha, s)
            rng = np.random.RandomState(s + 7000)
            X = rng.randn(DIM, BATCH_SIZE) * 0.3
            Y = target @ X
            w = init_weights(s + 5000)
            grads = compute_gradients(w, X, Y)
            er = np.mean([effective_rank(G) for G in grads])
            eff_ranks.append(er)
        mean_eff_rank = np.mean(eff_ranks)
        eff_rank_frac = mean_eff_rank / DIM

        # Anisotropy
        target_er = effective_rank(make_anisotropic_target(alpha, seeds[0]))
        target_sv = np.linalg.svd(make_anisotropic_target(alpha, seeds[0]), compute_uv=False)
        anisotropy = target_sv[0] / (target_sv[-1] + 1e-15)

        print(f"    Gradient eff rank: {mean_eff_rank:.1f}/{DIM} ({eff_rank_frac*100:.0f}%)")
        print(f"    Target anisotropy: {anisotropy:.1f}")

        # LR sweep
        best = {}
        for opt, grid in [('sgd', LR_SGD), ('muon', LR_MUON)]:
            best_lr, best_loss = grid[-1], float('inf')
            for lr in grid:
                losses = []
                for s in seeds[:3]:
                    target = make_anisotropic_target(alpha, s)
                    rng = np.random.RandomState(s + 7000)
                    X = rng.randn(DIM, BATCH_SIZE) * 0.3
                    Y = target @ X
                    w = init_weights(s + 5000)
                    fl = train(w, X, Y, lr, opt)
                    losses.append(fl)
                ml = np.mean([l for l in losses if np.isfinite(l)] or [float('inf')])
                if ml < best_loss:
                    best_loss = ml
                    best_lr = lr
            best[opt] = best_lr

        # Full eval
        sgd_losses, muon_losses = [], []
        for s in seeds:
            target = make_anisotropic_target(alpha, s)
            rng = np.random.RandomState(s + 7000)
            X = rng.randn(DIM, BATCH_SIZE) * 0.3
            Y = target @ X
            w = init_weights(s + 5000)
            sgd_losses.append(train(w, X, Y, best['sgd'], 'sgd'))
            w = init_weights(s + 5000)
            muon_losses.append(train(w, X, Y, best['muon'], 'muon'))

        sgd_mean = np.mean([l for l in sgd_losses if np.isfinite(l)] or [float('inf')])
        muon_mean = np.mean([l for l in muon_losses if np.isfinite(l)] or [float('inf')])
        adv = sgd_mean / max(muon_mean, 1e-30)

        results[alpha] = {
            'eff_rank': mean_eff_rank, 'eff_rank_frac': eff_rank_frac,
            'anisotropy': anisotropy, 'advantage': adv,
            'sgd': sgd_mean, 'muon': muon_mean,
        }
        print(f"    Advantage: {adv:.1f}x")

    # =========================================================================
    # RESULTS
    # =========================================================================
    print(f"\n\n{'='*100}")
    print("RESULTS: MUON ADVANTAGE vs EFFECTIVE RANK")
    print(f"{'='*100}")

    print(f"\n  {'alpha':>6}  {'Eff Rank %':>12}  {'Anisotropy':>12}  {'Advantage':>12}")
    print("  " + "-" * 46)
    for alpha in ALPHA_VALUES:
        r = results[alpha]
        print(f"  {alpha:>6.1f}  {r['eff_rank_frac']*100:>11.0f}%  {r['anisotropy']:>12.1f}  {r['advantage']:>12.1f}x")

    # Find peak
    advs = [results[a]['advantage'] for a in ALPHA_VALUES]
    fracs = [results[a]['eff_rank_frac'] for a in ALPHA_VALUES]
    peak_idx = np.argmax(advs)
    peak_frac = fracs[peak_idx]
    peak_alpha = ALPHA_VALUES[peak_idx]

    print(f"\n  Peak advantage: {advs[peak_idx]:.1f}x at alpha={peak_alpha} "
          f"(eff_rank_frac={peak_frac*100:.0f}%)")

    t1 = 0.2 < peak_frac < 0.8
    print(f"\n  T1: Peak at 20-80% effective rank?  --> {'PASS' if t1 else 'FAIL'}  ({peak_frac*100:.0f}%)")

    # Check for inverted-U shape
    if peak_idx > 0 and peak_idx < len(advs) - 1:
        t2 = advs[peak_idx] > advs[0] and advs[peak_idx] > advs[-1]
        print(f"  T2: Inverted-U shape (peak > both extremes)?  --> {'PASS' if t2 else 'FAIL'}")
    else:
        t2 = False
        print(f"  T2: Peak at edge, cannot confirm inverted-U  --> FAIL")

    print(f"\n{'='*100}")
    print("EXPERIMENT COMPLETE")
    print(f"{'='*100}")

In [ ]:
if __name__ == '__main__':
    main()

### Interpretation of Results

**Reading the output above:**

- The results table shows each $\alpha$ value's effective rank fraction, target anisotropy
  (condition number), and Muon advantage ratio.
- **T1** checks whether the peak advantage occurs at moderate effective rank (20-80%).
  If the peak is at an extreme (very low or very high effective rank), the inverted-U
  prediction fails.
- **T2** checks for the inverted-U shape: the peak must be strictly greater than both the
  most isotropic ($\alpha = 0.1$) and most anisotropic ($\alpha = 10.0$) conditions.

**Physical intuition for the sweet spot:**

At moderate effective rank, the gradient carries information in ~10-20 singular directions.
SGD processes all of them but with highly unequal step sizes (proportional to $\sigma_i$),
so it makes fast progress along $\sigma_1$ but neglects $\sigma_{15}$. Muon equalizes all
of them, making balanced progress across all informative directions. This is maximally
beneficial when there are *many* informative directions at *different* scales.

At low effective rank (high $\alpha$), the gradient is essentially rank-1. Muon still
equalizes, but now it is inflating 31 noise directions to the same magnitude as the one
signal direction -- the update becomes 97% noise.

At high effective rank (low $\alpha$), the gradient is already nearly isotropic. Muon's
equalization barely changes the direction, so advantage $\approx 1.0$.

## Conclusions

### What This Experiment Tests

**H22a** tests the non-trivial prediction that Muon's benefit has an **inverted-U shape**
as a function of gradient effective rank -- peaking at moderate anisotropy and declining
at both extremes. This is qualitatively different from standard preconditioning, where
more ill-conditioning always means more benefit.

### Why the Inverted-U Is a Strong Prediction

Standard preconditioners (Adam, natural gradient, KFAC) have a monotonic relationship
with conditioning: the worse the condition number, the more they help. The inverted-U
prediction is unique to Muon's Newton-Schulz mechanism because:

1. Muon does not rescale individual directions -- it **projects** to the nearest orthogonal matrix
2. This projection is maximally useful when there is structured anisotropy (multiple
   informative directions at different scales)
3. But harmful when the gradient is near-rank-1 (the projection amplifies noise)

### Implications

- **If T1+T2 pass:** Muon has an optimal "operating regime" -- moderate anisotropy --
  and practitioners should be aware that extremely anisotropic or isotropic gradients
  reduce its effectiveness.
- **If T1+T2 fail (monotonic increase):** Muon behaves like a standard preconditioner,
  and the "noise amplification" concern does not materialize in practice.
- **If T1+T2 fail (monotonic decrease or flat):** Anisotropy is not the relevant
  variable, and Muon's mechanism operates through a different channel.

### Connection to Prior Results

Prior experiments (H17) found that gradient anisotropy *positively* correlates with
Muon advantage. H22a asks whether this relationship continues monotonically or turns
over at extreme anisotropy. The two results together map out the full landscape of
Muon's spectral equalization benefit.